# Automated Barcoding Detection

Note to self: scriptify the manual line segment extraction, which may be useful

E2E file
→ select B-scan
→ obtain BM boundary
→ flatten to BM
→ crop fixed depth below BM
→ normalize ROI
→ return arrays + metadata

## E2E Prep, Flattening, and Cropping

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.barcode.data import (
    inspect_volume_layers,
    load_e2e_volume,
    preprocess_bscan,
)

# ------------------------------------------------------------------
# Preprocessing settings
# ------------------------------------------------------------------

NORMALIZE = False

# Normalization settings (only used when NORMALIZE=True)
LOWER_PERCENTILE = 1.0
UPPER_PERCENTILE = 99.0

# Supported:
#   "whole_roi": The entire matrix
#   "local_band": An interval within the ROI
NORMALIZATION_SCOPE = "whole_roi"

# Only used when NORMALIZATION_SCOPE == "local_band"
NORMALIZATION_CENTER_ROW = None
NORMALIZATION_MARGIN = 20

# Crop settings
DEPTH_BELOW_BM = 150

In [ ]:
E2E_PATH = PROJECT_ROOT / "data" / "heyex" / "meta" / "ea8.E2E"

volume = load_e2e_volume(E2E_PATH)

print("Volume type:", type(volume))
print("Volume shape:", volume.shape)
print("Available layers:", inspect_volume_layers(volume))

In [ ]:
# Select the central B-scan
NUM_BSCANS = len(volume)
BSCAN_INDEX = NUM_BSCANS // 2

bscan = volume[BSCAN_INDEX]

print(f"Number of B-scans: {NUM_BSCANS}")
print(f"Selected central B-scan: {BSCAN_INDEX}")
print(f"B-scan shape: {bscan.shape}")
print("Metadata:")
print(bscan.meta)

In [ ]:
processed = preprocess_bscan(
    volume=volume,
    bscan_index=BSCAN_INDEX,
    bm_layer_name="BM",
    depth_below_bm=DEPTH_BELOW_BM,
    normalize=NORMALIZE,
    normalization_scope=NORMALIZATION_SCOPE,
    normalization_center_row=NORMALIZATION_CENTER_ROW,
    normalization_margin=NORMALIZATION_MARGIN,
    lower_percentile=LOWER_PERCENTILE,
    upper_percentile=UPPER_PERCENTILE,
)

print("Raw shape:", processed.raw_bscan.shape)
print("BM boundary shape:", processed.bm_boundary.shape)
print("Flattened shape:", processed.flattened_bscan.shape)
print("Sub-BM crop shape:", processed.sub_bm_crop.shape)
print("Reference row:", processed.reference_row)
print("Normalization enabled:", processed.normalization_enabled)

if processed.normalization_enabled:
    print("Normalization scope:", processed.normalization_scope)
    print(
        "Normalization percentiles:",
        processed.lower_percentile,
        processed.upper_percentile,
    )
    print(
        "Normalization values:",
        processed.normalization_lower_value,
        processed.normalization_upper_value,
    )

    if processed.normalization_scope == "local_band":
        print(
            "Normalization band:",
            (
                processed.normalization_band_start,
                processed.normalization_band_end,
            ),
        )

In [ ]:
print("BM minimum row:", processed.bm_boundary.min())
print("BM maximum row:", processed.bm_boundary.max())
print("BM median row:", np.median(processed.bm_boundary))
print("First 10 coordinates:", processed.bm_boundary[:10])

In [ ]:
bm_layer = volume.layers["BM"]

print("Type:", type(bm_layer))
print("Attributes:", [name for name in dir(bm_layer) if not name.startswith("_")])

if hasattr(bm_layer, "data"):
    print("Data type:", type(bm_layer.data))
    print("Data shape:", np.asarray(bm_layer.data).shape)

In [ ]:
# Flattened scan
plt.figure(figsize=(12, 5))

plt.imshow(
    processed.flattened_bscan,
    cmap="gray",
    aspect="auto",
)

plt.axhline(
    processed.reference_row,
    linestyle="--",
    linewidth=1.5,
    color="yellow",
    label="Reference row",
)

plt.title(f"B-scan {processed.bscan_index} flattened to BM")
plt.xlabel("Horizontal position")
plt.ylabel("Axial position")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 7),
    sharex=True,
)

axes[0].imshow(
    processed.sub_bm_crop,
    cmap="gray",
    aspect="auto",
)

axes[0].set_title("Original sub-BM crop")
axes[0].set_ylabel("Depth below BM")

axes[1].imshow(
    processed.normalized_crop,
    cmap="gray",
    aspect="auto",
)

if processed.normalization_enabled:

    title = (
        "Normalized sub-BM crop\n"
        f"Scope: {processed.normalization_scope} | "
        f"{processed.lower_percentile:g}–"
        f"{processed.upper_percentile:g} percentiles"
    )

    if processed.normalization_scope == "local_band":
        title += (
            f"\nBand rows: "
            f"{processed.normalization_band_start}–"
            f"{processed.normalization_band_end}"
        )

    axes[1].set_title(title)

else:
    axes[1].set_title("Normalization disabled")

axes[1].set_xlabel("Horizontal position")
axes[1].set_ylabel("Depth below BM")

plt.tight_layout()
plt.show()

I actually suspect vertical persistence is going to end up being the key feature rather than brightness. The human eye is picking up those long, coherent bright columns—not just bright spots—and our algorithm should try to quantify exactly that. That also aligns well with the paper you’re using as inspiration: reduce the OCT to a robust 1D signal along the horizontal axis, segment contiguous abnormal regions, and then perform all measurements within those detected intervals.

In [ ]:
# Region below BM
plt.figure(figsize=(12, 4))

plt.imshow(
    processed.normalized_crop,
    cmap="gray",
    aspect="auto",
)

if processed.normalization_enabled:

    title = (
        f"Normalized region below BM "
        f"(depth={processed.depth_below_bm} px, "
        f"scope={processed.normalization_scope}, "
        f"percentiles={processed.lower_percentile:g}–"
        f"{processed.upper_percentile:g})"
    )

else:

    title = (
        f"Unnormalized region below BM "
        f"(depth={processed.depth_below_bm} px)"
    )

plt.title(title)
plt.xlabel("Horizontal position")
plt.ylabel("Depth below BM")

plt.tight_layout()
plt.show()

## Barcode Detection Tuning

* Implement ImageJ-like profile, where a line segment is made on the processed scan and the intensity is extracted
* Should have arguments for:
    * start: Where the line segment starts (default: 73, 135)
    * depth: How far below the top the starting point is
    * end: Where the line ends (default: 1169, 135)
    * stepsize: How many pixels each measurement is taken (default: 1 px)
    * plot: whether plot is saved or not (default: True)
    * data: Whether numerical data is extracted or not (default: True)
* The plot should be have gray scale values of 0-300 so regardless of the scan, the axis scale the same
* Should be able to take in a bscan


In [ ]:
from src.barcode.model_threshold import extract_intensity_profile

PROFILE_DEPTH = 50

profile = extract_intensity_profile(
    bscan=processed.normalized_crop,
    start=(0, PROFILE_DEPTH),
    depth=PROFILE_DEPTH,
    end=(processed.normalized_crop.shape[1] - 1, PROFILE_DEPTH),
    stepsize=1,
    profile_margin=0,          # Single-pixel profile (original behavior)
    aggregation="none",
    plot=True,
    data=True,
    overlay=True,              # Has no visible effect when profile_margin=0
    # plot_path=PROJECT_ROOT / "results" / "profile_plot.png",
    # data_path=PROJECT_ROOT / "results" / "profile_values.csv",
)

plt.show()

* Normaliza around the selected line instead of the whole scan, or average along PROFILE_DEPTH, +-2, +-4
* Overlay the profile plot onto the processed scan, and add marking utilities for manual marking of Barcoding, EA, and normal
* Do a series of sensitivity tests like different normalization values, different margins, and different stepsizes

In [ ]:
print(profile.profile_data.shape)
print(profile.profile_data[:10])

# Profile Overlay & Manual Annotation

This interactive visualization combines the profile plot and the processed scan. The profile is overlaid on the processed sub-Bruch's membrane (BM) region and shares a common horizontal axis, so we can see how well the signal and the data match.

To establish a reference for future automated detection methods, manual annotation utilities are provided for marking contiguous horizontal regions as:

- **Normal**
- **Early Atrophy (EA)**
- **Barcoding**

These annotations serve as exploratory ground truth for evaluating gradient-based methods, signal-processing approaches, and sequence models developed later in the project. Annotation intervals are stored alongside scan metadata for subsequent sensitivity analyses and algorithm validation.

In [ ]:
# %pip install ipympl
%matplotlib widget

from src.barcode.manual_annotation import create_manual_annotator

# Manual annotation settings
SAVE_ANNOTATIONS = True
OVERWRITE_ANNOTATIONS = False

ANNOTATION_OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "manual_annotations"
)

ANNOTATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ANNOTATION_OUTPUT_PATH = (
    ANNOTATION_OUTPUT_DIR
    / (
        f"{E2E_PATH.stem}_"
        f"bscan_{BSCAN_INDEX:03d}_"
        "annotations.json"
    )
)

LABEL_COLORS = {
    "Normal": "tab:green",
    "Early Atrophy (EA)": "tab:orange",
    "Barcoding": "tab:red",
}

SOURCE_METADATA = {
    "source_name": E2E_PATH.name,
    "source_path": str(E2E_PATH),
    "bscan_index": BSCAN_INDEX,
    "volume_shape": tuple(volume.shape),
    "processed_image_shape": tuple(
        processed.normalized_crop.shape
    ),
    "bm_layer_name": "BM",
    "reference_row": processed.reference_row,
    "depth_below_bm": processed.depth_below_bm,
    "normalization": processed.metadata["normalization"],
}

In [ ]:
# Overlay and manual annotation
annotator = create_manual_annotator(
    image=processed.normalized_crop,
    profile=profile,
    label_colors=LABEL_COLORS,
    source_metadata=SOURCE_METADATA,
    save=SAVE_ANNOTATIONS,
    output_path=ANNOTATION_OUTPUT_PATH,
    overwrite=OVERWRITE_ANNOTATIONS,
    figure_size=(14, 8),
    gray_value_limits=(0, 300),
)

annotator.show()

* Select Normal, Early Atrophy (EA), or Barcoding.
* Drag horizontally across either the scan or the profile.
* Use Undo last or Clear all as needed.
* Use Save JSON when SAVE_ANNOTATIONS = True.

In [ ]:
# Inspect current annotations
print(
    f"Number of annotated intervals: "
    f"{len(annotator.intervals)}"
)

for index, interval in enumerate(
    annotator.intervals,
    start=1,
):
    print(
        f"{index:02d}. "
        f"{interval.label}: "
        f"x={interval.x_start:.1f}–"
        f"{interval.x_end:.1f}, "
        f"width={interval.width_pixels:.1f} px"
    )

In [ ]:
# Annotation summary
annotation_summary = annotator.summary()

for label, values in annotation_summary.items():
    print(label)
    print(
        "  Interval count:",
        values["interval_count"],
    )
    print(
        "  Total annotated width:",
        f"{values['total_width_pixels']:.1f} px",
    )

In [ ]:
# Optional programmatic save
if SAVE_ANNOTATIONS:
    saved_path = annotator.save(
        overwrite=OVERWRITE_ANNOTATIONS,
    )

    print(
        "Annotations saved to:",
        saved_path,
    )
else:
    print(
        "Saving is disabled. Annotations remain available "
        "through annotator.intervals and annotator.to_dict()."
    )

# Sensitivity Tests

We now assess the robustness of the extraction pipeline should be evaluated under reasonable parameter variations.

The following experiments investigate how changes in profile extraction, normalization strategy, and sampling parameters affect the extracted intensity profile and derived measurements. These studies help identify parameter settings that produce stable and reproducible profiles while minimizing sensitivity to arbitrary implementation choices.

Sensitivity analyses are performed in three stages:

1. **Profile Extraction Expansion** — varying the spatial averaging strategy used to construct the profile.
2. **Normalization Scopes** — evaluating different intensity normalization approaches.
3. **Parameter Sensitivity Analysis** — systematically varying one parameter at a time and through factorial combinations.

## Profile Extraction Expansion

The baseline implementation extracts an intensity profile from a single horizontal row located at the selected profile depth. While computationally simple, individual rows may be sensitive to speckle noise and local anatomical variation.

To improve robustness, symmetric depth averaging is investigated by averaging neighboring rows surrounding the selected profile depth.

The following extraction strategies are evaluated:

| Method | Description |
|---------|-------------|
| Single Line | Baseline profile extracted from the selected depth only |
| Mean ±2 px | Mean intensity across   `PROFILE_DEPTH` ±2 pixels |
| Mean ±4 px | Mean intensity across `PROFILE_DEPTH` ±4 pixels |
| Median ±2 px | Median intensity across `PROFILE_DEPTH` ±2 pixels |
| Median ±4 px | Median intensity across `PROFILE_DEPTH` ±4 pixels |
| Average ± $\varepsilon$ px | Median intensity across `PROFILE_DEPTH` ±$\varepsilon$ pixels |



In [ ]:
# Reload updated profile-extraction code
import importlib
import src.barcode.model_threshold as model_threshold

importlib.reload(model_threshold)

extract_intensity_profile = (
    model_threshold.extract_intensity_profile
)

In [ ]:
# Profile extraction experiment settings
PROFILE_STEP_SIZE = 1.0

# Fully customizable sensitivity setting
EPSILON_MARGIN = 6
EPSILON_AGGREGATION = "median"

if EPSILON_MARGIN < 0:
    raise ValueError(
        "EPSILON_MARGIN must be greater than or equal to zero."
    )

if EPSILON_AGGREGATION not in {"mean", "median"}:
    raise ValueError(
        "EPSILON_AGGREGATION must be either 'mean' or 'median'."
    )

PROFILE_EXTRACTION_CONFIGS = {
    "Single line": {
        "profile_margin": 0,
        "aggregation": "none",
    },
    "Mean ±2 px": {
        "profile_margin": 2,
        "aggregation": "mean",
    },
    "Mean ±4 px": {
        "profile_margin": 4,
        "aggregation": "mean",
    },
    "Median ±2 px": {
        "profile_margin": 2,
        "aggregation": "median",
    },
    "Median ±4 px": {
        "profile_margin": 4,
        "aggregation": "median",
    },
    (
        f"{EPSILON_AGGREGATION.title()} "
        f"±{EPSILON_MARGIN} px"
    ): {
        "profile_margin": EPSILON_MARGIN,
        "aggregation": EPSILON_AGGREGATION,
    },
}

PROFILE_EXTRACTION_CONFIGS

In [ ]:
# Run profile extraction experiments
profile_extraction_results = {}

for method_name, settings in PROFILE_EXTRACTION_CONFIGS.items():
    result = extract_intensity_profile(
        bscan=processed.normalized_crop,
        start=(0, PROFILE_DEPTH),
        depth=PROFILE_DEPTH,
        end=(
            processed.normalized_crop.shape[1] - 1,
            PROFILE_DEPTH,
        ),
        stepsize=PROFILE_STEP_SIZE,
        profile_margin=settings["profile_margin"],
        aggregation=settings["aggregation"],
        plot=False,
        data=True,
        overlay=False,
    )

    profile_extraction_results[method_name] = result

    print(
        f"{method_name}: "
        f"{result.gray_values.size} measurements, "
        f"margin={result.profile_margin}, "
        f"aggregation={result.aggregation}"
    )

In [ ]:
# Compare profile extraction strategies
plt.figure(figsize=(14, 7))

for method_name, result in profile_extraction_results.items():
    horizontal_position = (
        result.distance
        + result.start[0]
    )

    plt.plot(
        horizontal_position,
        result.gray_values,
        linewidth=1.2,
        label=method_name,
    )

plt.xlim(
    -0.5,
    processed.normalized_crop.shape[1] - 0.5,
)

plt.ylim(0, 300)

plt.title(
    "Intensity profiles under alternative depth-aggregation strategies"
)
plt.xlabel("Horizontal position")
plt.ylabel("Gray value")
plt.grid(alpha=0.25)
plt.legend(
    loc="upper right",
    ncol=2,
)

plt.tight_layout()
plt.show()

In [ ]:
# Difference from the single-line baseline
baseline_result = profile_extraction_results[
    "Single line"
]

baseline_values = np.asarray(
    baseline_result.gray_values,
    dtype=np.float32,
)

baseline_x = (
    baseline_result.distance
    + baseline_result.start[0]
)

plt.figure(figsize=(14, 7))

for method_name, result in profile_extraction_results.items():
    if method_name == "Single line":
        continue

    difference = (
        np.asarray(
            result.gray_values,
            dtype=np.float32,
        )
        - baseline_values
    )

    plt.plot(
        baseline_x,
        difference,
        linewidth=1.1,
        label=method_name,
    )

plt.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

plt.xlim(
    -0.5,
    processed.normalized_crop.shape[1] - 0.5,
)

plt.title(
    "Profile differences relative to the single-line baseline"
)
plt.xlabel("Horizontal position")
plt.ylabel("Gray-value difference")
plt.grid(alpha=0.25)
plt.legend(
    loc="upper right",
    ncol=2,
)

plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison
import pandas as pd

profile_summary_records = []

for method_name, result in profile_extraction_results.items():
    values = np.asarray(
        result.gray_values,
        dtype=np.float32,
    )

    first_difference = np.diff(values)

    correlation_with_baseline = float(
        np.corrcoef(
            baseline_values,
            values,
        )[0, 1]
    )

    mean_absolute_difference = float(
        np.mean(
            np.abs(
                values - baseline_values
            )
        )
    )

    root_mean_squared_difference = float(
        np.sqrt(
            np.mean(
                (
                    values
                    - baseline_values
                )
                ** 2
            )
        )
    )

    profile_summary_records.append(
        {
            "method": method_name,
            "profile_margin": result.profile_margin,
            "band_height": (
                2 * result.profile_margin + 1
            ),
            "aggregation": result.aggregation,
            "number_of_measurements": values.size,
            "mean_gray_value": float(
                values.mean()
            ),
            "std_gray_value": float(
                values.std()
            ),
            "min_gray_value": float(
                values.min()
            ),
            "max_gray_value": float(
                values.max()
            ),
            "gradient_energy": float(
                np.mean(
                    first_difference**2
                )
            ),
            "correlation_with_single_line": (
                correlation_with_baseline
            ),
            "mean_absolute_difference": (
                mean_absolute_difference
            ),
            "root_mean_squared_difference": (
                root_mean_squared_difference
            ),
        }
    )

profile_extraction_summary = pd.DataFrame(
    profile_summary_records
)

profile_extraction_summary

In [ ]:
# Display the customizable epsilon extraction band
epsilon_method_name = (
    f"{EPSILON_AGGREGATION.title()} "
    f"±{EPSILON_MARGIN} px"
)

epsilon_profile = profile_extraction_results[
    epsilon_method_name
]

epsilon_display = extract_intensity_profile(
    bscan=processed.normalized_crop,
    start=(0, PROFILE_DEPTH),
    depth=PROFILE_DEPTH,
    end=(
        processed.normalized_crop.shape[1] - 1,
        PROFILE_DEPTH,
    ),
    stepsize=PROFILE_STEP_SIZE,
    profile_margin=EPSILON_MARGIN,
    aggregation=EPSILON_AGGREGATION,
    plot=True,
    data=True,
    overlay=True,
    overlay_color="cyan",
    overlay_alpha=0.25,
)

plt.show()

## Normalization Scopes

Intensity normalization can substantially influence the appearance of the extracted profile, particularly in regions containing large hypertransmission or local brightness variations.

Two normalization strategies are evaluated:

| Scope | Description |
|--------|-------------|
| Whole ROI | Percentile normalization computed using all pixels within the processed sub-BM region |
| Local Band | Percentile normalization computed only from the rows used for profile extraction |

Local-band normalization may better preserve relative intensity differences near the extracted profile while reducing the influence of distant anatomical structures. Whole-ROI normalization provides a consistent reference across the entire processed scan. using 0 and 100 as bounds is essentially max-min normalization

The extracted profiles obtained from each normalization strategy are compared visually and quantitatively.

In [ ]:
# Reload updated preprocessing code
import importlib

import src.barcode.data as barcode_data

importlib.reload(barcode_data)

preprocess_bscan = barcode_data.preprocess_bscan

In [ ]:
# Normalization-scope experiment settings
# Primary robust-percentile setting
NORMALIZATION_LOWER_PERCENTILE = 1.0
NORMALIZATION_UPPER_PERCENTILE = 99.0

# Include min-max normalization as a comparison
INCLUDE_MINMAX_COMPARISON = True

# Local normalization band centered on PROFILE_DEPTH
LOCAL_NORMALIZATION_MARGIN = 4

# Keep profile extraction fixed so only normalization changes
NORMALIZATION_PROFILE_MARGIN = 4
NORMALIZATION_PROFILE_AGGREGATION = "mean"
NORMALIZATION_PROFILE_STEP_SIZE = 1.0

if not (
    0
    <= NORMALIZATION_LOWER_PERCENTILE
    < NORMALIZATION_UPPER_PERCENTILE
    <= 100
):
    raise ValueError(
        "Normalization percentiles must satisfy "
        "0 <= lower < upper <= 100."
    )

if LOCAL_NORMALIZATION_MARGIN < 0:
    raise ValueError(
        "LOCAL_NORMALIZATION_MARGIN must be nonnegative."
    )

In [ ]:
# Normalization configurations
NORMALIZATION_CONFIGS = {
    (
        f"Whole ROI "
        f"({NORMALIZATION_LOWER_PERCENTILE:g}–"
        f"{NORMALIZATION_UPPER_PERCENTILE:g})"
    ): {
        "normalization_scope": "whole_roi",
        "normalization_center_row": None,
        "normalization_margin": 0,
        "lower_percentile": NORMALIZATION_LOWER_PERCENTILE,
        "upper_percentile": NORMALIZATION_UPPER_PERCENTILE,
    },
    (
        f"Local band ±{LOCAL_NORMALIZATION_MARGIN} px "
        f"({NORMALIZATION_LOWER_PERCENTILE:g}–"
        f"{NORMALIZATION_UPPER_PERCENTILE:g})"
    ): {
        "normalization_scope": "local_band",
        "normalization_center_row": PROFILE_DEPTH,
        "normalization_margin": LOCAL_NORMALIZATION_MARGIN,
        "lower_percentile": NORMALIZATION_LOWER_PERCENTILE,
        "upper_percentile": NORMALIZATION_UPPER_PERCENTILE,
    },
}

if INCLUDE_MINMAX_COMPARISON:
    NORMALIZATION_CONFIGS.update(
        {
            "Whole ROI min-max (0–100)": {
                "normalization_scope": "whole_roi",
                "normalization_center_row": None,
                "normalization_margin": 0,
                "lower_percentile": 0.0,
                "upper_percentile": 100.0,
            },
            (
                f"Local band ±{LOCAL_NORMALIZATION_MARGIN} px "
                "min-max (0–100)"
            ): {
                "normalization_scope": "local_band",
                "normalization_center_row": PROFILE_DEPTH,
                "normalization_margin": LOCAL_NORMALIZATION_MARGIN,
                "lower_percentile": 0.0,
                "upper_percentile": 100.0,
            },
        }
    )

NORMALIZATION_CONFIGS

In [ ]:
# Run normalization-scope experiments
normalization_processed_results = {}
normalization_profile_results = {}

for method_name, settings in NORMALIZATION_CONFIGS.items():

    current_processed = preprocess_bscan(
        volume=volume,
        bscan_index=BSCAN_INDEX,
        bm_layer_name="BM",
        depth_below_bm=DEPTH_BELOW_BM,
        normalize=True,
        normalization_scope=settings[
            "normalization_scope"
        ],
        normalization_center_row=settings[
            "normalization_center_row"
        ],
        normalization_margin=settings[
            "normalization_margin"
        ],
        lower_percentile=settings[
            "lower_percentile"
        ],
        upper_percentile=settings[
            "upper_percentile"
        ],
    )

    current_profile = extract_intensity_profile(
        bscan=current_processed.normalized_crop,
        start=(0, PROFILE_DEPTH),
        depth=PROFILE_DEPTH,
        end=(
            current_processed.normalized_crop.shape[1] - 1,
            PROFILE_DEPTH,
        ),
        stepsize=NORMALIZATION_PROFILE_STEP_SIZE,
        profile_margin=NORMALIZATION_PROFILE_MARGIN,
        aggregation=NORMALIZATION_PROFILE_AGGREGATION,
        plot=False,
        data=True,
    )

    normalization_processed_results[
        method_name
    ] = current_processed

    normalization_profile_results[
        method_name
    ] = current_profile

    print(method_name)
    print(
        "  Scope:",
        current_processed.normalization_scope,
    )
    print(
        "  Percentile intensity values:",
        current_processed.normalization_lower_value,
        current_processed.normalization_upper_value,
    )

    if (
        current_processed.normalization_scope
        == "local_band"
    ):
        print(
            "  Local band rows:",
            current_processed.normalization_band_start,
            "to",
            current_processed.normalization_band_end,
        )

In [ ]:
# Visual comparison of normalized sub-BM regions
number_of_methods = len(
    normalization_processed_results
)

fig, axes = plt.subplots(
    number_of_methods,
    1,
    figsize=(14, 3.2 * number_of_methods),
    sharex=True,
)

if number_of_methods == 1:
    axes = [axes]

for axis, (
    method_name,
    current_processed,
) in zip(
    axes,
    normalization_processed_results.items(),
):
    axis.imshow(
        current_processed.normalized_crop,
        cmap="gray",
        aspect="auto",
        vmin=0,
        vmax=1,
    )

    axis.axhline(
        PROFILE_DEPTH,
        linestyle="--",
        linewidth=1.2,
        label="Profile center",
    )

    if (
        current_processed.normalization_scope
        == "local_band"
    ):
        axis.axhspan(
            current_processed.normalization_band_start,
            current_processed.normalization_band_end,
            alpha=0.20,
            label="Normalization reference band",
        )

    axis.set_title(method_name)
    axis.set_ylabel("Depth below BM")
    axis.legend(loc="upper right")

axes[-1].set_xlabel("Horizontal position")

plt.tight_layout()
plt.show()

In [ ]:
# Profile comparison across normalization strategies
plt.figure(figsize=(14, 7))

for method_name, result in (
    normalization_profile_results.items()
):
    horizontal_position = (
        result.distance
        + result.start[0]
    )

    plt.plot(
        horizontal_position,
        result.gray_values,
        linewidth=1.2,
        label=method_name,
    )

plt.xlim(
    -0.5,
    processed.sub_bm_crop.shape[1] - 0.5,
)

plt.ylim(0, 300)

plt.title(
    "Intensity profiles under alternative normalization scopes"
)
plt.xlabel("Horizontal position")
plt.ylabel("Gray value")
plt.grid(alpha=0.25)
plt.legend(
    loc="upper right",
    ncol=2,
)

plt.tight_layout()
plt.show()

In [ ]:
# Quantitative normalization comparison
import pandas as pd

reference_name = (
    f"Whole ROI "
    f"({NORMALIZATION_LOWER_PERCENTILE:g}–"
    f"{NORMALIZATION_UPPER_PERCENTILE:g})"
)

reference_profile = np.asarray(
    normalization_profile_results[
        reference_name
    ].gray_values,
    dtype=np.float32,
)

normalization_summary_records = []

for method_name, current_profile in (
    normalization_profile_results.items()
):
    values = np.asarray(
        current_profile.gray_values,
        dtype=np.float32,
    )

    current_processed = (
        normalization_processed_results[
            method_name
        ]
    )

    difference = (
        values - reference_profile
    )

    normalization_summary_records.append(
        {
            "method": method_name,
            "scope": (
                current_processed.normalization_scope
            ),
            "lower_percentile": (
                current_processed.lower_percentile
            ),
            "upper_percentile": (
                current_processed.upper_percentile
            ),
            "normalization_margin": (
                current_processed.normalization_margin
            ),
            "band_start": (
                current_processed.normalization_band_start
            ),
            "band_end": (
                current_processed.normalization_band_end
            ),
            "lower_intensity_value": (
                current_processed.normalization_lower_value
            ),
            "upper_intensity_value": (
                current_processed.normalization_upper_value
            ),
            "mean_gray_value": float(
                values.mean()
            ),
            "std_gray_value": float(
                values.std()
            ),
            "dynamic_range": float(
                values.max() - values.min()
            ),
            "correlation_with_reference": float(
                np.corrcoef(
                    reference_profile,
                    values,
                )[0, 1]
            ),
            "mean_absolute_difference": float(
                np.mean(
                    np.abs(difference)
                )
            ),
            "root_mean_squared_difference": float(
                np.sqrt(
                    np.mean(
                        difference**2
                    )
                )
            ),
            "gradient_energy": float(
                np.mean(
                    np.diff(values) ** 2
                )
            ),
        }
    )

normalization_scope_summary = pd.DataFrame(
    normalization_summary_records
)

normalization_scope_summary

In [ ]:
# Differences from whole-ROI robust normalization
reference_result = (
    normalization_profile_results[
        reference_name
    ]
)

reference_x = (
    reference_result.distance
    + reference_result.start[0]
)

plt.figure(figsize=(14, 7))

for method_name, current_profile in (
    normalization_profile_results.items()
):
    if method_name == reference_name:
        continue

    difference = (
        np.asarray(
            current_profile.gray_values,
            dtype=np.float32,
        )
        - reference_profile
    )

    plt.plot(
        reference_x,
        difference,
        linewidth=1.1,
        label=method_name,
    )

plt.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

plt.xlim(
    -0.5,
    processed.sub_bm_crop.shape[1] - 0.5,
)

plt.title(
    "Profile differences relative to whole-ROI "
    "robust normalization"
)
plt.xlabel("Horizontal position")
plt.ylabel("Gray-value difference")
plt.grid(alpha=0.25)
plt.legend(
    loc="upper right",
)

plt.tight_layout()
plt.show()

## One-Factor Sensitivity Analysis

To evaluate the influence of individual processing parameters, a one-factor-at-a-time (OFAT) sensitivity analysis is performed. Each parameter is varied independently while all remaining parameters are held fixed at their baseline values.

Parameters investigated include:

- Profile extraction margin
- Aggregation method (mean vs. median)
- Normalization scope
- Normalization percentile bounds
- Sampling step size

This analysis identifies which parameters have the greatest influence on the extracted intensity profile and derived quantitative features.

In [ ]:
# One-factor-at-a-time sensitivity settings
OFAT_BASELINE = {
    "profile_margin": 4,
    "aggregation": "mean",
    "normalization_scope": "whole_roi",
    "normalization_margin": 4,
    "lower_percentile": 1.0,
    "upper_percentile": 99.0,
    "stepsize": 1.0,
}

OFAT_TEST_VALUES = {
    "profile_margin": [
        0,
        2,
        4,
        6,
        8,
    ],
    "aggregation": [
        "mean",
        "median",
    ],
    "normalization_scope": [
        "whole_roi",
        "local_band",
    ],
    "percentile_bounds": [
        (0.0, 100.0),
        (0.5, 99.5),
        (1.0, 99.0),
        (2.0, 98.0),
        (5.0, 95.0),
    ],
    "stepsize": [
        1.0,
        2.0,
        4.0,
        8.0,
    ],
}

OFAT_BASELINE

In [ ]:
# OFAT experiment runner
def run_profile_configuration(
    *,
    profile_margin: int,
    aggregation: str,
    normalization_scope: str,
    normalization_margin: int,
    lower_percentile: float,
    upper_percentile: float,
    stepsize: float,
):
    """
    Preprocess the central B-scan and extract one intensity profile
    using a completely specified parameter configuration.
    """

    if normalization_scope == "local_band":
        normalization_center_row = PROFILE_DEPTH
    elif normalization_scope == "whole_roi":
        normalization_center_row = None
    else:
        raise ValueError(
            "normalization_scope must be 'whole_roi' or 'local_band'."
        )

    current_processed = preprocess_bscan(
        volume=volume,
        bscan_index=BSCAN_INDEX,
        bm_layer_name="BM",
        depth_below_bm=DEPTH_BELOW_BM,
        normalize=True,
        normalization_scope=normalization_scope,
        normalization_center_row=normalization_center_row,
        normalization_margin=normalization_margin,
        lower_percentile=lower_percentile,
        upper_percentile=upper_percentile,
    )

    current_profile = extract_intensity_profile(
        bscan=current_processed.normalized_crop,
        start=(0, PROFILE_DEPTH),
        depth=PROFILE_DEPTH,
        end=(
            current_processed.normalized_crop.shape[1] - 1,
            PROFILE_DEPTH,
        ),
        stepsize=stepsize,
        profile_margin=profile_margin,
        aggregation=aggregation,
        plot=False,
        data=True,
        overlay=False,
    )

    return current_processed, current_profile

In [ ]:
# Baseline OFAT result
(
    ofat_baseline_processed,
    ofat_baseline_profile,
) = run_profile_configuration(
    **OFAT_BASELINE
)

OFAT_BASELINE_X = (
    np.asarray(
        ofat_baseline_profile.distance,
        dtype=np.float32,
    )
    + float(ofat_baseline_profile.start[0])
)

OFAT_BASELINE_VALUES = np.asarray(
    ofat_baseline_profile.gray_values,
    dtype=np.float32,
)

print("Baseline configuration:")
print(OFAT_BASELINE)
print()
print(
    "Number of baseline measurements:",
    OFAT_BASELINE_VALUES.size,
)

In [ ]:
# Construct OFAT configurations
ofat_configurations = []

# Profile extraction margin
for margin in OFAT_TEST_VALUES["profile_margin"]:
    config = OFAT_BASELINE.copy()
    config["profile_margin"] = margin

    ofat_configurations.append(
        {
            "factor": "profile_margin",
            "level": margin,
            "label": f"Margin ±{margin} px",
            "config": config,
        }
    )

# Aggregation method
for method in OFAT_TEST_VALUES["aggregation"]:
    config = OFAT_BASELINE.copy()
    config["aggregation"] = method

    ofat_configurations.append(
        {
            "factor": "aggregation",
            "level": method,
            "label": method.title(),
            "config": config,
        }
    )

# Normalization scope
for scope in OFAT_TEST_VALUES["normalization_scope"]:
    config = OFAT_BASELINE.copy()
    config["normalization_scope"] = scope

    ofat_configurations.append(
        {
            "factor": "normalization_scope",
            "level": scope,
            "label": scope.replace("_", " ").title(),
            "config": config,
        }
    )

# Normalization percentile bounds
for lower, upper in OFAT_TEST_VALUES["percentile_bounds"]:
    config = OFAT_BASELINE.copy()
    config["lower_percentile"] = lower
    config["upper_percentile"] = upper

    ofat_configurations.append(
        {
            "factor": "percentile_bounds",
            "level": (lower, upper),
            "label": f"{lower:g}–{upper:g}",
            "config": config,
        }
    )

# Sampling step size
for stepsize in OFAT_TEST_VALUES["stepsize"]:
    config = OFAT_BASELINE.copy()
    config["stepsize"] = stepsize

    ofat_configurations.append(
        {
            "factor": "stepsize",
            "level": stepsize,
            "label": f"{stepsize:g} px",
            "config": config,
        }
    )

print(
    "Number of OFAT configurations:",
    len(ofat_configurations),
)

In [ ]:
# Run OFAT experiments
ofat_results = []

for experiment in ofat_configurations:
    current_processed, current_profile = (
        run_profile_configuration(
            **experiment["config"]
        )
    )

    ofat_results.append(
        {
            **experiment,
            "processed": current_processed,
            "profile": current_profile,
        }
    )

    print(
        f"{experiment['factor']}: "
        f"{experiment['label']} "
        f"({current_profile.gray_values.size} measurements)"
    )

In [ ]:
# Align profiles for quantitative comparison
def interpolate_profile_to_baseline(
    profile_result,
) -> np.ndarray:
    current_x = (
        np.asarray(
            profile_result.distance,
            dtype=np.float32,
        )
        + float(profile_result.start[0])
    )

    current_values = np.asarray(
        profile_result.gray_values,
        dtype=np.float32,
    )

    aligned_values = np.interp(
        OFAT_BASELINE_X,
        current_x,
        current_values,
    )

    return aligned_values.astype(np.float32)

In [ ]:
# Calculate OFAT summary metrics
import pandas as pd


def safe_correlation(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    if (
        np.std(first) == 0
        or np.std(second) == 0
    ):
        return np.nan

    return float(
        np.corrcoef(
            first,
            second,
        )[0, 1]
    )


ofat_summary_records = []

for result in ofat_results:
    profile_result = result["profile"]
    processed_result = result["processed"]
    config = result["config"]

    aligned_values = interpolate_profile_to_baseline(
        profile_result
    )

    difference = (
        aligned_values
        - OFAT_BASELINE_VALUES
    )

    first_gradient = np.gradient(
        aligned_values
    )

    second_gradient = np.gradient(
        first_gradient
    )

    ofat_summary_records.append(
        {
            "factor": result["factor"],
            "level": str(result["level"]),
            "label": result["label"],
            "profile_margin": config["profile_margin"],
            "aggregation": config["aggregation"],
            "normalization_scope": config[
                "normalization_scope"
            ],
            "normalization_margin": config[
                "normalization_margin"
            ],
            "lower_percentile": config[
                "lower_percentile"
            ],
            "upper_percentile": config[
                "upper_percentile"
            ],
            "stepsize": config["stepsize"],
            "number_of_measurements": int(
                profile_result.gray_values.size
            ),
            "mean_gray_value": float(
                aligned_values.mean()
            ),
            "std_gray_value": float(
                aligned_values.std()
            ),
            "dynamic_range": float(
                aligned_values.max()
                - aligned_values.min()
            ),
            "gradient_energy": float(
                np.mean(
                    first_gradient**2
                )
            ),
            "curvature_energy": float(
                np.mean(
                    second_gradient**2
                )
            ),
            "correlation_with_baseline": safe_correlation(
                OFAT_BASELINE_VALUES,
                aligned_values,
            ),
            "mean_absolute_difference": float(
                np.mean(
                    np.abs(difference)
                )
            ),
            "root_mean_squared_difference": float(
                np.sqrt(
                    np.mean(
                        difference**2
                    )
                )
            ),
            "maximum_absolute_difference": float(
                np.max(
                    np.abs(difference)
                )
            ),
            "normalization_lower_value": (
                processed_result.normalization_lower_value
            ),
            "normalization_upper_value": (
                processed_result.normalization_upper_value
            ),
        }
    )

ofat_summary = pd.DataFrame(
    ofat_summary_records
)

ofat_summary

In [ ]:
# Visualize OFAT profiles by factor
OFAT_FACTOR_TITLES = {
    "profile_margin": "Profile extraction margin",
    "aggregation": "Aggregation method",
    "normalization_scope": "Normalization scope",
    "percentile_bounds": "Normalization percentile bounds",
    "stepsize": "Sampling step size",
}

for factor, factor_title in OFAT_FACTOR_TITLES.items():
    factor_results = [
        result
        for result in ofat_results
        if result["factor"] == factor
    ]

    plt.figure(
        figsize=(14, 6)
    )

    for result in factor_results:
        current_profile = result["profile"]

        current_x = (
            np.asarray(
                current_profile.distance,
                dtype=np.float32,
            )
            + current_profile.start[0]
        )

        plt.plot(
            current_x,
            current_profile.gray_values,
            linewidth=1.2,
            label=result["label"],
        )

    plt.xlim(
        -0.5,
        processed.sub_bm_crop.shape[1] - 0.5,
    )

    plt.ylim(
        0,
        300,
    )

    plt.title(
        f"OFAT sensitivity: {factor_title}"
    )

    plt.xlabel(
        "Horizontal position"
    )

    plt.ylabel(
        "Gray value"
    )

    plt.grid(
        alpha=0.25
    )

    plt.legend(
        loc="upper right",
        ncol=2,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# OFAT deviations from baseline
for factor, factor_title in OFAT_FACTOR_TITLES.items():
    factor_results = [
        result
        for result in ofat_results
        if result["factor"] == factor
    ]

    plt.figure(
        figsize=(14, 6)
    )

    for result in factor_results:
        aligned_values = (
            interpolate_profile_to_baseline(
                result["profile"]
            )
        )

        difference = (
            aligned_values
            - OFAT_BASELINE_VALUES
        )

        plt.plot(
            OFAT_BASELINE_X,
            difference,
            linewidth=1.1,
            label=result["label"],
        )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=1,
    )

    plt.xlim(
        -0.5,
        processed.sub_bm_crop.shape[1] - 0.5,
    )

    plt.title(
        f"OFAT deviation from baseline: {factor_title}"
    )

    plt.xlabel(
        "Horizontal position"
    )

    plt.ylabel(
        "Gray-value difference"
    )

    plt.grid(
        alpha=0.25
    )

    plt.legend(
        loc="upper right",
        ncol=2,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# Summarize sensitivity by factor
ofat_factor_summary = (
    ofat_summary
    .groupby(
        "factor",
        as_index=False,
    )
    .agg(
        mean_absolute_difference=(
            "mean_absolute_difference",
            "mean",
        ),
        maximum_mean_absolute_difference=(
            "mean_absolute_difference",
            "max",
        ),
        mean_rmse=(
            "root_mean_squared_difference",
            "mean",
        ),
        maximum_rmse=(
            "root_mean_squared_difference",
            "max",
        ),
        minimum_correlation=(
            "correlation_with_baseline",
            "min",
        ),
        gradient_energy_range=(
            "gradient_energy",
            lambda values: (
                values.max()
                - values.min()
            ),
        ),
    )
    .sort_values(
        "mean_absolute_difference",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

ofat_factor_summary

In [ ]:
# Relative influence of each processing factor
plot_table = (
    ofat_factor_summary
    .sort_values(
        "mean_absolute_difference",
        ascending=True,
    )
)

plt.figure(
    figsize=(10, 5)
)

plt.barh(
    plot_table["factor"],
    plot_table["mean_absolute_difference"],
)

plt.title(
    "Average profile sensitivity by processing factor"
)

plt.xlabel(
    "Mean absolute difference from baseline"
)

plt.ylabel(
    "Processing factor"
)

plt.tight_layout()
plt.show()

## Factorial Sensitivity Analysis

While one-factor analyses isolate the effect of individual parameters, interactions between parameters may also influence profile extraction.

A factorial sensitivity analysis is therefore performed by evaluating combinations of multiple parameter settings. This experiment assesses the robustness of the extraction pipeline across a broader parameter space and identifies combinations that consistently produce stable intensity profiles.

The resulting profiles and quantitative summary statistics are compared to determine parameter settings that provide reliable performance while maintaining sensitivity to clinically relevant features such as EA and barcoding.

In [ ]:
# Factorial sensitivity settings
FACTORIAL_GRID = {
    "profile_margin": [
        2,
        4,
        6,
    ],
    "aggregation": [
        "mean",
        "median",
    ],
    "normalization_scope": [
        "whole_roi",
        "local_band",
    ],
    "percentile_bounds": [
        (0.0, 100.0),
        (1.0, 99.0),
        (2.0, 98.0),
    ],
    "stepsize": [
        1.0,
        2.0,
    ],
}

# Used only when normalization_scope == "local_band"
FACTORIAL_NORMALIZATION_MARGIN = 4

FACTORIAL_GRID

In [ ]:
# Calculate factorial experiment size
from math import prod

factorial_level_counts = {
    parameter: len(values)
    for parameter, values in FACTORIAL_GRID.items()
}

NUMBER_OF_FACTORIAL_CONFIGURATIONS = prod(
    factorial_level_counts.values()
)

print("Levels per factor:")
for parameter, count in factorial_level_counts.items():
    print(f"  {parameter}: {count}")

print(
    "\nTotal factorial configurations:",
    NUMBER_OF_FACTORIAL_CONFIGURATIONS,
)

In [ ]:
# Generate factorial configurations
from itertools import product

factorial_configurations = []

for (
    profile_margin,
    aggregation,
    normalization_scope,
    percentile_bounds,
    stepsize,
) in product(
    FACTORIAL_GRID["profile_margin"],
    FACTORIAL_GRID["aggregation"],
    FACTORIAL_GRID["normalization_scope"],
    FACTORIAL_GRID["percentile_bounds"],
    FACTORIAL_GRID["stepsize"],
):
    lower_percentile, upper_percentile = (
        percentile_bounds
    )

    configuration = {
        "profile_margin": profile_margin,
        "aggregation": aggregation,
        "normalization_scope": normalization_scope,
        "normalization_margin": (
            FACTORIAL_NORMALIZATION_MARGIN
        ),
        "lower_percentile": lower_percentile,
        "upper_percentile": upper_percentile,
        "stepsize": stepsize,
    }

    configuration_id = (
        f"margin_{profile_margin}_"
        f"{aggregation}_"
        f"{normalization_scope}_"
        f"pct_{lower_percentile:g}_{upper_percentile:g}_"
        f"step_{stepsize:g}"
    )

    factorial_configurations.append(
        {
            "configuration_id": configuration_id,
            "config": configuration,
        }
    )

print(
    "Generated configurations:",
    len(factorial_configurations),
)

factorial_configurations[:3]

In [ ]:
# Run factorial sensitivity experiment
factorial_results = []

for index, experiment in enumerate(
    factorial_configurations,
    start=1,
):
    current_processed, current_profile = (
        run_profile_configuration(
            **experiment["config"]
        )
    )

    factorial_results.append(
        {
            "configuration_id": (
                experiment["configuration_id"]
            ),
            "config": experiment["config"],
            "processed": current_processed,
            "profile": current_profile,
        }
    )

    if (
        index == 1
        or index % 10 == 0
        or index == len(factorial_configurations)
    ):
        print(
            f"Completed {index}/"
            f"{len(factorial_configurations)} configurations"
        )

In [ ]:
# Calculate factorial sensitivity metrics
factorial_summary_records = []

for result in factorial_results:
    config = result["config"]
    profile_result = result["profile"]
    processed_result = result["processed"]

    aligned_values = interpolate_profile_to_baseline(
        profile_result
    )

    difference = (
        aligned_values
        - OFAT_BASELINE_VALUES
    )

    first_gradient = np.gradient(
        aligned_values
    )

    second_gradient = np.gradient(
        first_gradient
    )

    factorial_summary_records.append(
        {
            "configuration_id": (
                result["configuration_id"]
            ),
            "profile_margin": (
                config["profile_margin"]
            ),
            "band_height": (
                2 * config["profile_margin"] + 1
            ),
            "aggregation": (
                config["aggregation"]
            ),
            "normalization_scope": (
                config["normalization_scope"]
            ),
            "normalization_margin": (
                config["normalization_margin"]
            ),
            "lower_percentile": (
                config["lower_percentile"]
            ),
            "upper_percentile": (
                config["upper_percentile"]
            ),
            "stepsize": (
                config["stepsize"]
            ),
            "number_of_measurements": int(
                profile_result.gray_values.size
            ),
            "mean_gray_value": float(
                aligned_values.mean()
            ),
            "std_gray_value": float(
                aligned_values.std()
            ),
            "dynamic_range": float(
                aligned_values.max()
                - aligned_values.min()
            ),
            "gradient_energy": float(
                np.mean(
                    first_gradient**2
                )
            ),
            "curvature_energy": float(
                np.mean(
                    second_gradient**2
                )
            ),
            "correlation_with_baseline": (
                safe_correlation(
                    OFAT_BASELINE_VALUES,
                    aligned_values,
                )
            ),
            "mean_absolute_difference": float(
                np.mean(
                    np.abs(difference)
                )
            ),
            "root_mean_squared_difference": float(
                np.sqrt(
                    np.mean(
                        difference**2
                    )
                )
            ),
            "maximum_absolute_difference": float(
                np.max(
                    np.abs(difference)
                )
            ),
            "normalization_lower_value": (
                processed_result
                .normalization_lower_value
            ),
            "normalization_upper_value": (
                processed_result
                .normalization_upper_value
            ),
        }
    )

factorial_summary = pd.DataFrame(
    factorial_summary_records
)

factorial_summary.head()

In [ ]:
# Rank configurations by stability
factorial_ranked = (
    factorial_summary
    .sort_values(
        by=[
            "root_mean_squared_difference",
            "mean_absolute_difference",
            "correlation_with_baseline",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

factorial_ranked[
    [
        "configuration_id",
        "profile_margin",
        "aggregation",
        "normalization_scope",
        "lower_percentile",
        "upper_percentile",
        "stepsize",
        "correlation_with_baseline",
        "mean_absolute_difference",
        "root_mean_squared_difference",
        "gradient_energy",
    ]
].head(15)

In [ ]:
# Configurations with the largest profile changes
factorial_most_different = (
    factorial_summary
    .sort_values(
        "root_mean_squared_difference",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

factorial_most_different[
    [
        "configuration_id",
        "profile_margin",
        "aggregation",
        "normalization_scope",
        "lower_percentile",
        "upper_percentile",
        "stepsize",
        "correlation_with_baseline",
        "mean_absolute_difference",
        "root_mean_squared_difference",
        "maximum_absolute_difference",
    ]
].head(15)

In [ ]:
# Average sensitivity by parameter level
factorial_level_summaries = {}

FACTORIAL_FACTORS = [
    "profile_margin",
    "aggregation",
    "normalization_scope",
    "lower_percentile",
    "upper_percentile",
    "stepsize",
]

for factor in FACTORIAL_FACTORS:
    summary = (
        factorial_summary
        .groupby(
            factor,
            as_index=False,
        )
        .agg(
            mean_correlation=(
                "correlation_with_baseline",
                "mean",
            ),
            minimum_correlation=(
                "correlation_with_baseline",
                "min",
            ),
            mean_absolute_difference=(
                "mean_absolute_difference",
                "mean",
            ),
            mean_rmse=(
                "root_mean_squared_difference",
                "mean",
            ),
            maximum_rmse=(
                "root_mean_squared_difference",
                "max",
            ),
            mean_gradient_energy=(
                "gradient_energy",
                "mean",
            ),
            mean_curvature_energy=(
                "curvature_energy",
                "mean",
            ),
        )
        .sort_values(
            factor
        )
        .reset_index(
            drop=True
        )
    )

    factorial_level_summaries[
        factor
    ] = summary

    print(f"\nFactor: {factor}")
    display(summary)

In [ ]:
# Two-factor interaction summaries
INTERACTION_PAIRS = [
    (
        "profile_margin",
        "aggregation",
    ),
    (
        "normalization_scope",
        "lower_percentile",
    ),
    (
        "normalization_scope",
        "profile_margin",
    ),
    (
        "profile_margin",
        "stepsize",
    ),
    (
        "aggregation",
        "normalization_scope",
    ),
]

factorial_interaction_summaries = {}

for first_factor, second_factor in INTERACTION_PAIRS:
    interaction_name = (
        f"{first_factor}_x_{second_factor}"
    )

    interaction_summary = (
        factorial_summary
        .groupby(
            [
                first_factor,
                second_factor,
            ],
            as_index=False,
        )
        .agg(
            mean_rmse=(
                "root_mean_squared_difference",
                "mean",
            ),
            mean_absolute_difference=(
                "mean_absolute_difference",
                "mean",
            ),
            mean_correlation=(
                "correlation_with_baseline",
                "mean",
            ),
            mean_gradient_energy=(
                "gradient_energy",
                "mean",
            ),
        )
        .sort_values(
            "mean_rmse"
        )
        .reset_index(
            drop=True
        )
    )

    factorial_interaction_summaries[
        interaction_name
    ] = interaction_summary

    print(
        f"\nInteraction: "
        f"{first_factor} × {second_factor}"
    )

    display(
        interaction_summary
    )

In [ ]:
# Interaction heatmaps
SELECTED_HEATMAPS = [
    (
        "profile_margin",
        "aggregation",
    ),
    (
        "normalization_scope",
        "lower_percentile",
    ),
    (
        "profile_margin",
        "stepsize",
    ),
]

for row_factor, column_factor in SELECTED_HEATMAPS:
    heatmap_table = (
        factorial_summary
        .pivot_table(
            index=row_factor,
            columns=column_factor,
            values="root_mean_squared_difference",
            aggfunc="mean",
        )
    )

    figure, axis = plt.subplots(
        figsize=(8, 5)
    )

    image = axis.imshow(
        heatmap_table.values,
        aspect="auto",
    )

    axis.set_xticks(
        np.arange(
            heatmap_table.shape[1]
        )
    )

    axis.set_xticklabels(
        heatmap_table.columns
    )

    axis.set_yticks(
        np.arange(
            heatmap_table.shape[0]
        )
    )

    axis.set_yticklabels(
        heatmap_table.index
    )

    axis.set_xlabel(
        column_factor.replace(
            "_",
            " ",
        ).title()
    )

    axis.set_ylabel(
        row_factor.replace(
            "_",
            " ",
        ).title()
    )

    axis.set_title(
        "Mean profile RMSE: "
        f"{row_factor.replace('_', ' ').title()} × "
        f"{column_factor.replace('_', ' ').title()}"
    )

    figure.colorbar(
        image,
        ax=axis,
        label="Mean RMSE from baseline",
    )

    for row_index in range(
        heatmap_table.shape[0]
    ):
        for column_index in range(
            heatmap_table.shape[1]
        ):
            value = heatmap_table.iloc[
                row_index,
                column_index,
            ]

            axis.text(
                column_index,
                row_index,
                f"{value:.2f}",
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.show()

In [ ]:
# Compare representative factorial profiles
NUMBER_OF_REPRESENTATIVE_PROFILES = 3

representative_ids = []

# Most stable configurations
for configuration_id in (
    factorial_ranked[
        "configuration_id"
    ].head(
        NUMBER_OF_REPRESENTATIVE_PROFILES
    )
):
    if configuration_id not in representative_ids:
        representative_ids.append(
            configuration_id
        )

# Most different configurations
for configuration_id in (
    factorial_most_different[
        "configuration_id"
    ].head(
        NUMBER_OF_REPRESENTATIVE_PROFILES
    )
):
    if configuration_id not in representative_ids:
        representative_ids.append(
            configuration_id
        )

results_by_id = {
    result["configuration_id"]: result
    for result in factorial_results
}

plt.figure(
    figsize=(14, 7)
)

plt.plot(
    OFAT_BASELINE_X,
    OFAT_BASELINE_VALUES,
    linewidth=2,
    label="Baseline",
)

for configuration_id in representative_ids:
    current_result = results_by_id[
        configuration_id
    ]

    current_profile = current_result[
        "profile"
    ]

    current_x = (
        np.asarray(
            current_profile.distance,
            dtype=np.float32,
        )
        + current_profile.start[0]
    )

    plt.plot(
        current_x,
        current_profile.gray_values,
        linewidth=1.0,
        alpha=0.85,
        label=configuration_id,
    )

plt.xlim(
    -0.5,
    processed.sub_bm_crop.shape[1] - 0.5,
)

plt.ylim(
    0,
    300,
)

plt.title(
    "Representative profiles from the factorial experiment"
)

plt.xlabel(
    "Horizontal position"
)

plt.ylabel(
    "Gray value"
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="upper right",
    fontsize=8,
)

plt.tight_layout()
plt.show()

In [ ]:
# Stability and signal-preservation ranking
BASELINE_GRADIENT_ENERGY = float(
    np.mean(
        np.gradient(
            OFAT_BASELINE_VALUES
        )
        ** 2
    )
)

factorial_ranked_for_signal = (
    factorial_summary
    .copy()
)

factorial_ranked_for_signal[
    "gradient_energy_ratio"
] = (
    factorial_ranked_for_signal[
        "gradient_energy"
    ]
    / (
        BASELINE_GRADIENT_ENERGY
        + 1e-8
    )
)

factorial_ranked_for_signal[
    "gradient_energy_deviation"
] = np.abs(
    factorial_ranked_for_signal[
        "gradient_energy_ratio"
    ]
    - 1.0
)

# Standardize the two quantities before combining them.
rmse_mean = (
    factorial_ranked_for_signal[
        "root_mean_squared_difference"
    ].mean()
)

rmse_std = (
    factorial_ranked_for_signal[
        "root_mean_squared_difference"
    ].std()
    + 1e-8
)

gradient_deviation_mean = (
    factorial_ranked_for_signal[
        "gradient_energy_deviation"
    ].mean()
)

gradient_deviation_std = (
    factorial_ranked_for_signal[
        "gradient_energy_deviation"
    ].std()
    + 1e-8
)

factorial_ranked_for_signal[
    "stability_signal_score"
] = (
    (
        factorial_ranked_for_signal[
            "root_mean_squared_difference"
        ]
        - rmse_mean
    )
    / rmse_std
    +
    (
        factorial_ranked_for_signal[
            "gradient_energy_deviation"
        ]
        - gradient_deviation_mean
    )
    / gradient_deviation_std
)

factorial_ranked_for_signal = (
    factorial_ranked_for_signal
    .sort_values(
        "stability_signal_score"
    )
    .reset_index(
        drop=True
    )
)

factorial_ranked_for_signal[
    [
        "configuration_id",
        "profile_margin",
        "aggregation",
        "normalization_scope",
        "lower_percentile",
        "upper_percentile",
        "stepsize",
        "root_mean_squared_difference",
        "correlation_with_baseline",
        "gradient_energy_ratio",
        "stability_signal_score",
    ]
].head(15)

The two final rankings answer slightly different questions:

* factorial_ranked identifies configurations closest to the selected baseline.
* factorial_ranked_for_signal identifies configurations that are stable while retaining similar spatial-gradient structure, which is more relevant when preserving barcode-like intensity variation.